# Football Market Predictor

**Hypothesis:** Pinnacle prices odds based on long-term team reputation. When ELO, recent form, and xG signal teams are evenly matched but reputations differ, the market misprices home wins and draws — especially in lower-visibility leagues where the American-skewed audience relies on historical prestige.

**Data:** 73,469 matches × 12 European leagues (2006–2026). xG for top-5 leagues from Understat. Pinnacle closing line as benchmark.

**Validation:** Walk-forward — train on all past seasons, test on the next one. 19 folds, no leakage.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import brier_score_loss
from sklearn.calibration import calibration_curve
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

HOLDOUT_YEAR = 2019  # selection: <2019, holdout: >=2019

In [ ]:
features = pd.read_parquet('../data/processed/football_features.parquet')
results  = pd.read_parquet('../data/results/walkforward_results.parquet')
results_np = pd.read_parquet('../data/results/walkforward_results_no_pinnacle.parquet')
fi_with    = pd.read_csv('../data/results/feature_importance_full.csv', index_col=0).squeeze()
fi_without = pd.read_csv('../data/results/feature_importance_no_pinnacle.csv', index_col=0).squeeze()

results['Date'] = pd.to_datetime(results['Date'])
results['year'] = results['Season'].str[:4].astype(int)
results_np['year'] = results_np['Season'].str[:4].astype(int)

print(f'Matches: {len(features):,}  |  Leagues: {features["League"].nunique()}  |  Seasons: {features["Season"].nunique()}')
print(f'Walk-forward rows: {len(results):,}')
print(f'Date range: {results["Date"].min().date()} — {results["Date"].max().date()}')

## 1. Feature Coverage

85 features across all leagues. xG data covers only top-5 leagues (Understat). H2H and shot data have partial coverage — filled with 0 during training.

In [ ]:
from src.football_model import FEATURE_COLS
avail = [c for c in FEATURE_COLS if c in features.columns]
coverage = features[avail].notna().mean().sort_values()

fig, ax = plt.subplots(figsize=(10, 9))

colors = ['#d62728' if v < 0.5 else '#ff7f0e' if v < 0.8 else '#2ca02c' for v in coverage]
bars = ax.barh(range(len(coverage)), coverage.values, color=colors, height=0.7)
ax.set_yticks(range(len(coverage)))
ax.set_yticklabels(coverage.index, fontsize=8)
ax.axvline(0.8, color='gray', linestyle='--', linewidth=1, alpha=0.7, label='80% threshold')
ax.set_xlabel('Fraction non-null')
ax.set_title(f'Feature Coverage — {len(avail)} features across 73k matches')

from matplotlib.patches import Patch
legend_els = [
    Patch(color='#2ca02c', label=f'≥80%  ({(coverage>=0.8).sum()} features)'),
    Patch(color='#ff7f0e', label=f'50–80% ({((coverage>=0.5)&(coverage<0.8)).sum()} features) — shots, H2H'),
    Patch(color='#d62728', label=f'<50%   ({(coverage<0.5).sum()} features) — xG (top-5 only)'),
]
ax.legend(handles=legend_els, loc='lower right')
ax.set_xlim(0, 1.05)
plt.tight_layout()
plt.show()

print(f'xG features (< 50% coverage): only top-5 leagues have Understat data')
print(f'H2H features (~70%): requires historical match pairs — normal for early seasons')

## 2. Calibration — Model vs Pinnacle

Calibration curve compares predicted probability to actual frequency. A well-calibrated model follows the diagonal. The model overestimates in absolute terms (ECE ≈ 0.12), but edge is measured **relative to Pinnacle** — so absolute miscalibration does not affect strategy returns.

In [ ]:
# Use P1 for calibration — the final strategy league
p1 = results[results['League'] == 'P1'].dropna(subset=['pred_home','pred_draw','pred_away','target','market_home'])

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
outcomes = [
    ('Home Win',  'pred_home',  'market_home',  'H'),
    ('Draw',      'pred_draw',  'market_draw',  'D'),
    ('Away Win',  'pred_away',  'market_away',  'A'),
]

for ax, (label, pred_col, mkt_col, code) in zip(axes, outcomes):
    y_true = (p1['target'] == code).astype(int)

    prob_true_m, prob_pred_m = calibration_curve(y_true, p1[pred_col], n_bins=8, strategy='quantile')
    prob_true_p, prob_pred_p = calibration_curve(y_true, p1[mkt_col],  n_bins=8, strategy='quantile')

    ax.plot([0, 1], [0, 1], '--', color='gray', linewidth=1, alpha=0.6, label='Perfect')
    ax.plot(prob_pred_p, prob_true_p, 'o--', color='#d62728', linewidth=1.5, markersize=5, label='Pinnacle')
    ax.plot(prob_pred_m, prob_true_m, 's-',  color='#1f77b4', linewidth=2,   markersize=6, label='Model')

    ax.set_title(label)
    ax.set_xlabel('Predicted probability')
    ax.set_ylabel('Actual frequency')
    ax.legend(fontsize=9)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)

fig.suptitle('Calibration: Primeira Liga (Portugal) — Model vs Pinnacle', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# ECE
def ece(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins+1)
    ece_val = 0
    for i in range(n_bins):
        mask = (y_prob >= bins[i]) & (y_prob < bins[i+1])
        if mask.sum() == 0: continue
        ece_val += mask.sum() * abs(y_true[mask].mean() - y_prob[mask].mean())
    return ece_val / len(y_true)

for label, pred_col, _, code in outcomes:
    y_true = (p1['target'] == code).astype(int).values
    e = ece(y_true, p1[pred_col].values)
    print(f'{label:<12} ECE = {e:.3f}')

## 3. ROI by Season — Primeira Liga (Portugal), home+draw, edge ≥ 0.18

Walk-forward holdout: 2019–2026. Selection period (2006–2018) used to identify the strategy. Positive in 13 of 18 seasons overall.

In [ ]:
p1_bets = results[
    (results['League'] == 'P1') &
    (results['best_outcome'].isin(['home', 'draw'])) &
    (results['best_edge'] >= 0.18) &
    (results['kelly_size'] > 0)
].copy()

season_stats = p1_bets.groupby('Season').agg(
    n_bets   = ('profit', 'count'),
    profit   = ('profit', 'sum'),
    staked   = ('kelly_size', 'sum'),
    year     = ('year', 'first'),
).assign(roi=lambda d: d['profit'] / d['staked'])

fig, ax = plt.subplots(figsize=(13, 5))

colors = ['#2ca02c' if (r > 0 and y >= HOLDOUT_YEAR) else
          '#98df8a' if (r > 0 and y < HOLDOUT_YEAR) else
          '#d62728' if (r <= 0 and y >= HOLDOUT_YEAR) else
          '#ff9896'
          for r, y in zip(season_stats['roi'], season_stats['year'])]

x = range(len(season_stats))
bars = ax.bar(x, season_stats['roi'] * 100, color=colors, width=0.75)
ax.axhline(0, color='black', linewidth=0.8)
ax.axvline(season_stats.index.tolist().index('2018/2019') - 0.5,
           color='navy', linewidth=1.5, linestyle='--', alpha=0.7)
ax.text(season_stats.index.tolist().index('2018/2019') - 0.5 + 0.1, ax.get_ylim()[1] * 0.92,
        'Holdout →', color='navy', fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels([s[:4] for s in season_stats.index], rotation=45)
ax.set_ylabel('ROI (%)')
ax.set_title('Primeira Liga (Portugal) — home+draw, edge ≥ 0.18 | ROI by season')

from matplotlib.patches import Patch
legend_els = [
    Patch(color='#2ca02c', label='Holdout positive'),
    Patch(color='#d62728', label='Holdout negative'),
    Patch(color='#98df8a', label='Selection positive'),
    Patch(color='#ff9896', label='Selection negative'),
]
ax.legend(handles=legend_els, ncol=4, fontsize=9, loc='upper left')

for i, (s, row) in enumerate(season_stats.iterrows()):
    ax.text(i, row['roi'] * 100 + (2 if row['roi'] >= 0 else -4),
            f"{row['n_bets']}", ha='center', fontsize=7.5, color='black')

plt.tight_layout()
plt.savefig("../results/figures/roi_by_season.png", dpi=150, bbox_inches="tight")
plt.show()

sel = season_stats[season_stats['year'] < HOLDOUT_YEAR]
hld = season_stats[season_stats['year'] >= HOLDOUT_YEAR]
print(f'Selection (2006–2018): ROI={sel["profit"].sum()/sel["staked"].sum():+.1%}, '
      f'{(sel["roi"]>0).sum()}/{len(sel)} positive seasons')
print(f'Holdout   (2019–2026): ROI={hld["profit"].sum()/hld["staked"].sum():+.1%}, '
      f'{(hld["roi"]>0).sum()}/{len(hld)} positive seasons')

## 4. Bankroll Simulation — Dynamic Kelly

Quarter-Kelly sizing with 10% bankroll cap per bet. Starting bankroll $1,000 over all 292 bets across 18 seasons. Max drawdown reached 70% — realistic for any Kelly-based system.

In [ ]:
p1_bets_sorted = p1_bets.sort_values('Date').copy()

START = 1000.0
bankroll = START
history = [bankroll]
dates   = [p1_bets_sorted['Date'].iloc[0]]

for _, row in p1_bets_sorted.iterrows():
    stake = min(row['kelly_size'] * bankroll, bankroll * 0.10)
    bankroll += row['profit'] * stake
    bankroll = max(bankroll, 1.0)
    history.append(bankroll)
    dates.append(row['Date'])

history = np.array(history)
running_max = np.maximum.accumulate(history)
drawdown = (history - running_max) / running_max * 100

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=False,
                                gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.35})

ax1.plot(range(len(history)), history, color='#1f77b4', linewidth=1.8)
ax1.axhline(START, color='gray', linestyle='--', linewidth=1, alpha=0.6)

holdout_start = (p1_bets_sorted['year'] >= HOLDOUT_YEAR).idxmax()
holdout_idx = p1_bets_sorted.index.get_loc(holdout_start) + 1
ax1.axvline(holdout_idx, color='navy', linestyle='--', linewidth=1.5, alpha=0.7)
ax1.text(holdout_idx + 2, max(history) * 0.95, 'Holdout →', color='navy', fontsize=10)

ax1.fill_between(range(len(history)), START, history,
                 where=np.array(history) > START, alpha=0.15, color='green')
ax1.fill_between(range(len(history)), START, history,
                 where=np.array(history) <= START, alpha=0.15, color='red')

ax1.set_ylabel('Bankroll ($)')
ax1.set_title(f'Bankroll simulation — Primeira Liga (Portugal), $1,000 start'
              f' → ${bankroll:.0f} final  |  Max drawdown: {drawdown.min():.0f}%')

ax2.fill_between(range(len(drawdown)), drawdown, 0, color='#d62728', alpha=0.6)
ax2.set_ylabel('Drawdown (%)')
ax2.set_xlabel('Bet number')
ax2.set_title('Drawdown')

plt.tight_layout()
plt.show()

print(f'Final bankroll: ${bankroll:.0f}  ({(bankroll/START - 1)*100:+.0f}% return)')
print(f'Max drawdown:   {drawdown.min():.1f}%')
print(f'Total bets:     {len(p1_bets_sorted)}')
print(f'Avg bets/season: {len(p1_bets_sorted)/season_stats["year"].nunique():.1f}')

## 5. Pinnacle Leakage — Feature Importance Ablation

Pinnacle implied probabilities were included as model features, creating a potential leakage loop: the model learns to replicate Pinnacle rather than find independent signal. Feature importance shows Pinnacle columns at **~32% of total split gain**.

I ran the full pipeline a second time with Pinnacle columns removed entirely — same architecture, same Optuna budget, same walk-forward folds.

In [ ]:
TOP_N = 15
PINNACLE_COLS = {'implied_home', 'implied_draw', 'implied_away'}

top_with    = fi_with.head(TOP_N)
top_without = fi_without.head(TOP_N)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, fi, title in [
    (axes[0], top_with,    'With Pinnacle features'),
    (axes[1], top_without, 'Without Pinnacle features'),
]:
    colors = ['#d62728' if idx in PINNACLE_COLS else '#1f77b4' for idx in fi.index]
    ax.barh(range(len(fi)), fi.values, color=colors, height=0.7)
    ax.set_yticks(range(len(fi)))
    ax.set_yticklabels(fi.index, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel('Feature importance (split gain)')
    ax.set_title(title)

from matplotlib.patches import Patch
axes[0].legend(handles=[
    Patch(color='#d62728', label='Pinnacle features'),
    Patch(color='#1f77b4', label='Independent features'),
], fontsize=9)

pinnacle_sum = fi_with[list(PINNACLE_COLS)].sum()
fig.suptitle(f'Feature Importance: Pinnacle accounts for {pinnacle_sum:.0%} of split gain '
             f'→ without Pinnacle, ELO dominates at {fi_without.iloc[0]:.0%}',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig("../results/figures/feature_importance_ablation.png", dpi=150, bbox_inches="tight")
plt.show()

print(f'Pinnacle total importance: {pinnacle_sum:.1%}')
print(f'Without Pinnacle — top feature: {fi_without.index[0]} ({fi_without.iloc[0]:.1%})')

## 6. Edge Distribution — Where the Model Sees Mispricing

In [ ]:
# Edge = model_prob - pinnacle_implied_prob, for all P1 matches
p1_all = results[results['League'] == 'P1'].dropna(subset=['best_edge', 'best_outcome'])
p1_all_np = results_np[results_np['League'] == 'P1'].dropna(subset=['best_edge', 'best_outcome'])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, data, label, color in [
    (axes[0], p1_all,    'With Pinnacle features',    '#1f77b4'),
    (axes[1], p1_all_np, 'Without Pinnacle features', '#2ca02c'),
]:
    home_edge = data[data['best_outcome'] == 'home']['best_edge']
    draw_edge = data[data['best_outcome'] == 'draw']['best_edge']
    away_edge = data[data['best_outcome'] == 'away']['best_edge']

    bins = np.linspace(-0.3, 0.5, 40)
    ax.hist(home_edge, bins=bins, alpha=0.6, label='Home', color='#1f77b4')
    ax.hist(draw_edge, bins=bins, alpha=0.6, label='Draw', color='#ff7f0e')
    ax.hist(away_edge, bins=bins, alpha=0.6, label='Away', color='#2ca02c')
    ax.axvline(0.18, color='red', linestyle='--', linewidth=1.5, label='Edge threshold 0.18')
    ax.set_xlabel('Edge (model − Pinnacle)')
    ax.set_ylabel('Count')
    ax.set_title(f'P1 edge distribution\n{label}')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print('P1 bets above edge 0.18:')
for outcome in ['home', 'draw', 'away']:
    n_with = ((p1_all['best_outcome'] == outcome) & (p1_all['best_edge'] >= 0.18)).sum()
    n_wo   = ((p1_all_np['best_outcome'] == outcome) & (p1_all_np['best_edge'] >= 0.18)).sum()
    print(f'  {outcome:<5}  with Pinnacle: {n_with:>3}  |  without: {n_wo:>3}')

## 7. Strategy Comparison — All Evaluated Strategies

Grid search over ~500 combinations on selection data (2006–2018), validated on holdout (2019–2026). Most strategies collapse on holdout — multiple comparison inflation. The Deflated Sharpe Ratio corrects for this.

In [ ]:
LEAGUE_NAMES = {
    'P1': 'Primeira Liga (Portugal)',
    'D1': 'Bundesliga (Germany)',
    'G1': 'Super League (Greece)',
    'E1': 'Championship (England)',
    'N1': 'Eredivisie (Netherlands)',
    'E0': 'Premier League (England)',
    'SP1': 'La Liga (Spain)',
}

strategies = [
    # label, roi, sharpe, n_bets, p_value, marker
    ('P1 home+draw ≥0.18\n(with Pinnacle)',          0.138, 0.579, 292, 0.021, 'star',   '#2ca02c'),
    ('P1 home+draw ≥0.18\n(no Pinnacle)',             0.111, 0.387, 379, 0.060, 'circle', '#98df8a'),
    ('P1+G1+E1 home+draw ≥0.18\n(with Pinnacle)',     0.103, 0.099, 663, 0.006, 'circle', '#1f77b4'),
    ('D1 draw ≥0.25\n(with Pinnacle)',                0.319, 0.606,  55, 0.091, 'circle', '#ff7f0e'),
    ('D1 draw ≥0.25\n(no Pinnacle)',                 -0.105, None,   48, None,  'circle', '#d62728'),
    ('G1+P1 draw ≥0.15\n(no Pinnacle)',               0.184, 0.160, 218, 0.055, 'circle', '#9467bd'),
    ('G1+N1+P1 draw ≥0.18\n(no Pinnacle)',            0.059, 0.322, 178, 0.298, 'circle', '#8c564b'),
]

labels = [s[0] for s in strategies]
rois   = [s[1] * 100 for s in strategies]
sharpes = [s[2] if s[2] is not None else 0 for s in strategies]
n_bets = [s[3] for s in strategies]
pvals  = [s[4] for s in strategies]
colors = [s[6] for s in strategies]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ROI bar
x = range(len(strategies))
axes[0].barh(x, rois, color=colors, height=0.65)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_yticks(x)
axes[0].set_yticklabels(labels, fontsize=8.5)
axes[0].set_xlabel('Holdout ROI (%)')
axes[0].set_title('Holdout ROI by Strategy')
for i, (roi, nb, pv) in enumerate(zip(rois, n_bets, pvals)):
    txt = f'{roi:+.0f}%  ({nb} bets)'
    if pv is not None:
        txt += f'  p={pv:.3f}'
    axes[0].text(roi + (1 if roi >= 0 else -1), i, txt,
                va='center', ha='left' if roi >= 0 else 'right', fontsize=7.5)

# Sharpe scatter
valid_idx = [i for i, s in enumerate(strategies) if s[2] is not None]
axes[1].scatter([rois[i] for i in valid_idx], [sharpes[i] for i in valid_idx],
               c=[colors[i] for i in valid_idx], s=150, zorder=3)
axes[1].axvline(0, color='gray', linewidth=0.8, alpha=0.5)
axes[1].axhline(0, color='gray', linewidth=0.8, alpha=0.5)
axes[1].axhline(0.5, color='green', linestyle='--', linewidth=1, alpha=0.5, label='Sharpe 0.5')
for i in valid_idx:
    axes[1].annotate(labels[i], (rois[i], sharpes[i]),
                    textcoords='offset points', xytext=(5, 3), fontsize=7.5)
axes[1].set_xlabel('Holdout ROI (%)')
axes[1].set_ylabel('Sharpe Ratio')
axes[1].set_title('ROI vs Sharpe — only P1 survives both tests')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

## 8. Final Strategy Summary

In [ ]:
from scipy import stats

p1_holdout = p1_bets[p1_bets['year'] >= HOLDOUT_YEAR].copy()
p1_all_seasons = p1_bets.copy()

# Win rate vs market-implied win rate
win_rate = (p1_holdout['profit'] > 0).mean()
mkt_win_rate = p1_holdout.apply(
    lambda r: r['market_home'] if r['best_outcome'] == 'home' else r['market_draw'], axis=1
).mean()

# Binomial test
n_wins = (p1_holdout['profit'] > 0).sum()
binom = stats.binomtest(n_wins, len(p1_holdout), mkt_win_rate, alternative='greater')

# Sharpe across seasons
hld_stats = p1_holdout.groupby('Season').agg(
    profit=('profit', 'sum'), staked=('kelly_size', 'sum')
).assign(roi=lambda d: d['profit'] / d['staked'])
sharpe = hld_stats['roi'].mean() / hld_stats['roi'].std()

# Deflated Sharpe Ratio
N = 500  # strategies tested in grid search
T = len(hld_stats)  # number of seasons
skew = hld_stats['roi'].skew()
kurt = hld_stats['roi'].kurtosis()
sr_annualized = sharpe
# PSR: P(SR* > 0) where SR* is Sharpe of best strategy after multiple comparisons
# DSR approximation from Lopez de Prado (2018)
dsr_numerator = sharpe * np.sqrt(T - 1)
dsr_denominator = np.sqrt(T - 1 - skew * sharpe + (kurt - 1) / 4 * sharpe**2)
dsr = dsr_numerator / dsr_denominator if dsr_denominator > 0 else sharpe
dsr_corrected = dsr / np.sqrt(1 + np.log(N))
dsr_pval = 1 - stats.norm.cdf(dsr_corrected)

print('=' * 55)
print('Final Strategy: Primeira Liga (Portugal)')
print('Home wins + Draws, edge ≥ 0.18 vs Pinnacle')
print('=' * 55)
print(f'Holdout period:        2019–2026')
print(f'Holdout ROI:          {p1_holdout["profit"].sum()/p1_holdout["kelly_size"].sum():+.1%}')
print(f'Overall ROI (all 18 seasons): {p1_bets["profit"].sum()/p1_bets["kelly_size"].sum():+.1%}')
print(f'Sharpe Ratio:          {sharpe:.3f}')
print(f'Deflated Sharpe Ratio: {dsr_corrected:.3f}  (p = {dsr_pval:.3f}, N={N} strategies)')
print(f'Win rate vs market:    {win_rate:.1%} vs {mkt_win_rate:.1%}')
print(f'p-value (binomial):    {binom.pvalue:.3f}')
print(f'Positive seasons:      {(hld_stats["roi"]>0).sum()}/{len(hld_stats)} holdout')
print(f'Bets per season:       ~{len(p1_holdout)/len(hld_stats):.0f}')
print(f'Survives Pinnacle ablation: Yes — ROI +11.1% without Pinnacle features')
print()
print('Holdout season breakdown:')
for s, row in hld_stats.iterrows():
    print(f"  {s}  ROI={row['roi']:+.1%}  ({p1_holdout[p1_holdout['Season']==s].shape[0]} bets)")

## Decision Log

**Why Primeira Liga (Portugal) and not Bundesliga (Germany)?**  
Bundesliga draw showed +31.9% ROI **with** Pinnacle in features. Without Pinnacle — same strategy returned −10.5%. Pinnacle is extremely accurate on Bundesliga — the model was finding noise around its predictions, not real signal.

**Why not Super League (Greece)?**  
Greece draw without Pinnacle shows +12.4% ROI but p=0.19 and Sharpe 0.065. Seasonal breakdown: multiple seasons at −100%, one at +93%, another at +104%. The positive mean is driven by two outlier seasons. With only 8–9 bets/season, one cold streak eliminates years of gains.

**On the Pinnacle leakage:**  
Portugal's edge proved robust — ROI drops from +13.8% to +11.1% but stays positive and directionally consistent. The leakage partially helped (calibration), partially hurt (masked Greece signal), and created false edge in Bundesliga.

**Next step:** Paper trading on Primeira Liga — home wins and draws — edge ≥ 0.18, recording both opening and closing Pinnacle lines for CLV measurement.